# Notebook — Classical segmentation: k-means, GMM/EM, and an MRF prior

Segment a synthetic 3-tissue brain slice (CSF / GM / WM) three ways, each one a
step up:
- **k-means** — hard clustering of intensities (assign ↔ update).
- **GMM + EM** — soft memberships; handles different per-tissue spread; the
  E-step/M-step in plain NumPy, with the log-likelihood curve.
- **+ MRF / ICM** — add a Potts smoothness prior on the labels and clean up the
  salt-and-pepper mislabels (the Week-2 Bayesian idea, on discrete labels).

NumPy + Matplotlib only. The real thing on brain-MRI data is **Project 2**
(`project_segmentation.md`, the toned-down HMRF-GMM-EM of assignment `a3`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (10, 4)

# --- synthetic 3-tissue brain slice: concentric CSF / GM / WM inside a brain mask ---
N = 96
yy, xx = np.mgrid[0:N, 0:N]
r = np.sqrt((xx - 48)**2 + (yy - 48)**2)
true = np.full((N, N), -1)        # -1 = outside brain
true[r < 40] = 0                  # CSF
true[r < 32] = 1                  # GM
true[r < 20] = 2                  # WM
mask = true >= 0
means_true = np.array([0.30, 0.55, 0.80])
clean = np.where(mask, means_true[np.clip(true, 0, 2)], 0.0)
noisy = clean + 0.06 * rng.standard_normal((N, N))   # overlapping intensities after noise

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(clean, cmap='gray', vmin=0, vmax=1); ax[0].set_title('true tissues'); ax[0].axis('off')
ax[1].imshow(noisy, cmap='gray', vmin=0, vmax=1); ax[1].set_title('noisy observation y'); ax[1].axis('off')
plt.show()

vals = noisy[mask].reshape(-1, 1)    # intensities of in-brain voxels
print('in-brain voxels:', vals.shape[0])

## 1. K-means (hard clustering of intensities)

Alternate: **assign** each voxel to the nearest centre, then **update** each
centre to its voxels' mean. We sort the centres so label 0/1/2 = CSF/GM/WM.

In [ ]:
K = 3
x = vals[:, 0]
centers = np.quantile(x, [0.2, 0.5, 0.8])     # spread-out init
for _ in range(25):
    a = np.abs(x[:, None] - centers[None, :]).argmin(1)   # assign
    for k in range(K):
        if np.any(a == k):
            centers[k] = x[a == k].mean()                  # update
order = np.argsort(centers); centers = centers[order]
a = np.abs(x[:, None] - centers[None, :]).argmin(1)        # final labels, sorted

def to_image(labels):
    img = np.full((N, N), np.nan)
    img[mask] = labels
    return img

print('k-means centres (CSF, GM, WM):', np.round(centers, 3), ' true:', means_true)
plt.figure(figsize=(5, 5))
plt.imshow(to_image(a), cmap='viridis'); plt.title('k-means segmentation'); plt.axis('off')
plt.colorbar(ticks=[0, 1, 2]); plt.show()

## 2. GMM + EM (soft memberships)

Model intensities as a mixture of 3 Gaussians and fit by EM: the **E-step**
computes each voxel's posterior membership to each tissue; the **M-step** updates
each tissue's weight, mean, and standard deviation. The log-likelihood rises
every iteration.

In [ ]:
def gauss(x, m, s):
    return np.exp(-(x - m)**2 / (2*s**2)) / (np.sqrt(2*np.pi) * s)

mu = centers.copy()                 # init from k-means
sd = np.full(K, x.std())
w  = np.full(K, 1.0 / K)
lls = []
for it in range(60):
    # E-step: responsibilities (memberships)
    R = np.stack([w[k] * gauss(x, mu[k], sd[k]) for k in range(K)], axis=1)  # (Nvox, K)
    lls.append(np.sum(np.log(R.sum(1) + 1e-12)))
    R = R / (R.sum(1, keepdims=True) + 1e-12)
    # M-step
    Nk = R.sum(0)
    w  = Nk / Nk.sum()
    mu = (R * x[:, None]).sum(0) / Nk
    sd = np.sqrt((R * (x[:, None] - mu[None, :])**2).sum(0) / Nk) + 1e-3

hard = R.argmax(1)
print('GMM means:', np.round(mu, 3), ' sds:', np.round(sd, 3), ' weights:', np.round(w, 2))

fig, ax = plt.subplots(1, 4, figsize=(15, 4))
for k in range(K):
    m = np.full((N, N), np.nan); m[mask] = R[:, k]
    ax[k].imshow(m, cmap='magma', vmin=0, vmax=1); ax[k].set_title(f'membership: class {k}'); ax[k].axis('off')
ax[3].imshow(to_image(hard), cmap='viridis'); ax[3].set_title('GMM hard labels'); ax[3].axis('off')
plt.tight_layout(); plt.show()

plt.figure(figsize=(7, 3.5)); plt.plot(lls, 'o-'); plt.xlabel('EM iteration')
plt.ylabel('log-likelihood'); plt.title('EM increases the log-likelihood'); plt.show()

## 3. Add an MRF prior on the labels (ICM)

Intensities alone leave speckle. A **Potts** MRF prior says neighbours should
share a label; we run **ICM** — for each voxel pick the label maximising
(log-likelihood) + β·(number of agreeing neighbours). This is the Week-2 Bayesian
prior, applied to discrete labels; watch the speckle vanish.

In [ ]:
# per-voxel log-likelihood of each class (whole image)
loglik = np.stack([np.log(gauss(noisy, mu[k], sd[k]) + 1e-12) for k in range(K)], axis=-1)  # (N,N,K)

lab = np.full((N, N), -1); lab[mask] = hard
beta = 1.5
for sweep in range(6):
    nxt = lab.copy()
    for i in range(1, N-1):
        for j in range(1, N-1):
            if not mask[i, j]:
                continue
            best_k, best_score = -1, -1e18
            for k in range(K):
                agree = ((lab[i+1, j] == k) + (lab[i-1, j] == k)
                         + (lab[i, j+1] == k) + (lab[i, j-1] == k))
                score = loglik[i, j, k] + beta * agree
                if score > best_score:
                    best_score, best_k = score, k
            nxt[i, j] = best_k
    lab = nxt

icm = lab[mask]
fig, ax = plt.subplots(1, 3, figsize=(13, 4.5))
ax[0].imshow(to_image(hard), cmap='viridis'); ax[0].set_title('GMM only (speckled)'); ax[0].axis('off')
ax[1].imshow(to_image(icm),  cmap='viridis'); ax[1].set_title(f'GMM + MRF/ICM (beta={beta})'); ax[1].axis('off')
ax[2].imshow(np.where(mask, true, np.nan), cmap='viridis'); ax[2].set_title('ground truth'); ax[2].axis('off')
plt.tight_layout(); plt.show()

acc_gmm = np.mean(hard == true[mask]); acc_icm = np.mean(icm == true[mask])
print(f'label agreement with truth — GMM: {acc_gmm:.3f}   GMM+MRF: {acc_icm:.3f}')

## Exercises

1. Raise the noise (0.10, 0.15). Where do k-means and GMM start to fail, and does
   the MRF prior rescue them?
2. Sweep `beta` (0, 0.5, 1.5, 4). Too small leaves speckle; too large erases thin
   structures — find the sweet spot (exactly the `beta` tuning in assignment a3).
3. Add a smooth **bias field** (multiply the image by a low-frequency ramp) and
   watch single tissues split into two clusters — then think about how to estimate
   and remove it (the modified-FCM half of a3).
4. Compute per-class **Dice** instead of pixel agreement.
5. Initialise k-means randomly several times; observe convergence to different
   local optima.

**Next:** the kernel-methods notebook (`03_kernel_pca_toy.ipynb`) handles
clusters that are *not* blobs — where k-means/GMM struggle and a kernel helps. The
full segmentation project is **Project 2** in `week_6`.